# MNIST — обучение MLP (PyTorch)

Пайплайн: данные → модель → цикл обучения/теста → сохранение весов.

## 1. Импорты

In [14]:
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

MNIST_MEAN, MNIST_STD = 0.1307, 0.3081


def load_mnist():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((MNIST_MEAN,), (MNIST_STD,)),
    ])
    train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
    test_ds = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
    return train_ds, test_ds

## 2. Модель

MLP: два скрытых слоя с BatchNorm + Dropout, финальный слой на 10 классов.

In [15]:
class MLP(nn.Module):
    def __init__(self):
            super().__init__()
            self.fc1 = nn.Linear(28*28, 512)
            self.bn1 = nn.BatchNorm1d(512)
            self.fc2 = nn.Linear(512, 1024)
            self.bn2 = nn.BatchNorm1d(1024)
            self.fc3 = nn.Linear(1024, 512)
            self.bn3 = nn.BatchNorm1d(512)
            self.fc4 = nn.Linear(512, 256)
            self.bn4 = nn.BatchNorm1d(256)
            self.fc5 = nn.Linear(256, 10)
            self.dropout = nn.Dropout(0.3)
        
    def forward(self, x):
        x = x.flatten(1)
        x = F.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.dropout(x)
        x = F.relu(self.bn3(self.fc3(x)))
        x = self.dropout(x)
        x = F.relu(self.bn4(self.fc4(x)))
        x = self.dropout(x)
        return self.fc5(x)

## 3. Функции обучения и оценки

In [16]:
def train(model, device, loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        loss = F.cross_entropy(model(data), target)
        loss.backward()
        optimizer.step()
        if batch_idx % 200 == 0:
            print(f"epoch {epoch} [{batch_idx * len(data)}/{len(loader.dataset)}] loss: {loss.item():.4f}")

In [17]:
def test(model, device, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            pred = model(data).argmax(dim=1)
            correct += pred.eq(target).sum().item()
    acc = correct / len(loader.dataset)
    print(f"test accuracy: {acc:.4f}")
    return acc

## 4. Устройство (CUDA / MPS / CPU)

In [18]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("using device:", device)

using device: cuda


## 5. Данные

In [19]:
train_ds, test_ds = load_mnist()

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256)

## 6. Модель и оптимизатор

In [20]:
model = MLP().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## 7. Цикл обучения

In [21]:
total_start = time.time()
for epoch in range(1, 4):
    epoch_start = time.time()
    train(model, device, train_loader, optimizer, epoch)
    test(model, device, test_loader)
    print(f"epoch {epoch} took {time.time() - epoch_start:.2f}s")
print(f"total training time: {time.time() - total_start:.2f}s")

epoch 1 [0/60000] loss: 2.4290
epoch 1 [25600/60000] loss: 0.1397
epoch 1 [51200/60000] loss: 0.1311
test accuracy: 0.9688
epoch 1 took 13.14s
epoch 2 [0/60000] loss: 0.2253
epoch 2 [25600/60000] loss: 0.0817
epoch 2 [51200/60000] loss: 0.0873
test accuracy: 0.9744
epoch 2 took 13.04s
epoch 3 [0/60000] loss: 0.1814
epoch 3 [25600/60000] loss: 0.0423
epoch 3 [51200/60000] loss: 0.1460
test accuracy: 0.9805
epoch 3 took 13.06s
total training time: 39.24s


## 8. Сохранение модели

In [22]:
torch.save(model.state_dict(), "mnist_mlp.pt")
print("model saved to mnist_mlp.pt")

model saved to mnist_mlp.pt


## Сравнение с CPU

In [24]:

model_cpu = MLP().to('cpu')
optimizer_cpu = torch.optim.Adam(model_cpu.parameters(), lr=1e-3)

total_start = time.time()
for epoch in range(1, 4):
    epoch_start = time.time()
    train(model_cpu, 'cpu', train_loader, optimizer_cpu, epoch)
    acc = test(model_cpu, 'cpu', test_loader)
    print(f"эпоха {epoch}: точность={acc:.4f}, время={time.time() - epoch_start:.2f}с")
cpu_time = time.time() - total_start
print(f"Общее время CPU: {cpu_time:.2f}с")

epoch 1 [0/60000] loss: 2.4455
epoch 1 [25600/60000] loss: 0.2766
epoch 1 [51200/60000] loss: 0.1517
test accuracy: 0.9694
эпоха 1: точность=0.9694, время=20.65с
epoch 2 [0/60000] loss: 0.1255
epoch 2 [25600/60000] loss: 0.0899
epoch 2 [51200/60000] loss: 0.0750
test accuracy: 0.9733
эпоха 2: точность=0.9733, время=20.57с
epoch 3 [0/60000] loss: 0.1160
epoch 3 [25600/60000] loss: 0.1176
epoch 3 [51200/60000] loss: 0.0336
test accuracy: 0.9778
эпоха 3: точность=0.9778, время=20.42с
Общее время CPU: 61.65с
